# Sequential Monte Carlo

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

The SMC runner exposes the [`redist`](https://alarm-redist.org/redist/) sequential Monte Carlo
sampler. Unlike Rust ReCom and Forest ReCom, it does not advance one Markov chain by repeatedly
modifying the previous plan.
Through sequential splitting, it builds a population of weighted partial plans and ultimately
produces an ensemble with the diagnostics and weights used by `redist`.

## Input shapefile

`SMCRunnerConfig` identifies an ESRI shapefile by its stem, without the `.shp` extension. Keep
the `.shp`, `.shx`, `.dbf`, `.prj`, and any other sidecars together in the same directory. The
whole directory is mounted read-only so GDAL and `redist` can open the bundle.

The attribute table must contain the population and constraint columns named in the run. Geometry
and adjacency are read by the native workflow rather than supplied as a separate dual graph.

In [ ]:
from gerrytools.mgrp import SMCRunInfo, SMCRunnerConfig

config = SMCRunnerConfig(
    shapefile_path="data/precincts",
    output_folder="output",
    log_folder="logs",
)

## Configure the map and sampler

`SMCRunInfo` combines settings for two `redist` stages. `pop_col`, `n_dists`, and either
`pop_tol` or `pop_bounds` configure the map. The remaining fields configure the SMC sampler
and its output. Keeping them in one validated object makes the complete effective run available
before Docker starts.

In [ ]:
run = SMCRunInfo(
    pop_col="TOTPOP",
    n_dists=14,
    n_sims=1_000,
    pop_tol=0.01,
    compactness=1.0,
    tally_columns=["TOTPOP", "VAP", "BVAP"],
    writer="ben",
    rng_seed=2026,
)

Run the sampler once Docker is available and the shapefile bundle exists:

```python
from gerrytools.mgrp import RunContainer

with RunContainer(config) as container:
    output_path = container.run(run)
```

Diagnostic output from R and its native libraries is captured in the run log. `run()` verifies
the primary output and promised sidecars before returning `output_path`.

## Population bounds

The usual configuration supplies `pop_tol`, a relative tolerance around ideal district
population. When a project has explicit integer bounds, `pop_bounds` replaces that derived
interval with `[lower, target, upper]`. The three values must be nonnegative and ordered.

In [ ]:
bounded_run = SMCRunInfo(
    pop_col="TOTPOP",
    n_dists=14,
    n_sims=1_000,
    pop_bounds=[740_000, 750_000, 760_000],
    writer="ben",
    rng_seed=2026,
)

## Sampler controls

The `redist` defaults are preserved unless the run overrides them:

| Setting | Role | Default |
| --- | --- | --- |
| `n_sims` | Number of ensemble samples | Required |
| `compactness` | Strength of the sampler's compactness preference | `1.0` |
| `resample` | Perform the final resampling step | `False` |
| `adapt_k_thresh` | Threshold used to select the branching value at each split | `0.985` |
| `seq_alpha` | Weight adjustment at each resampling step | `0.5` |
| `pop_temper` | Strength of automatic population tempering | `0.0` |
| `final_infl` | Inflation of the final population constraint | `1.0` |
| `rng_seed` | Native random-number-generator seed | `42` |

`pop_temper` controls automatic population tempering, while `final_infl` loosens the population
constraint on the final split. Both change the effective algorithm. `verbose` enables
intermediate diagnostics; `silent` suppresses them. Any emitted diagnostics are written to the
run log.

## Add soft constraints

SMC constraints are weighted penalties rather than hard rejection rules. Positive strengths
downweight plans with larger penalty statistics. Several constraints can be chained in one
builder:

| Builder | Statistic being penalized |
| --- | --- |
| `group_hinge()` | District group shares below the nearest target |
| `group_power()` | Distance of district group shares from one or two targets |
| `status_quo()` | Population-weighted departure from a reference plan |
| `splits()` | Administrative units intersecting more than one district |

Constraint scales differ. A strength that is modest for one statistic may dominate another.
Calibrate each strength against its statistic and the sampler diagnostics.

In [ ]:
from gerrytools.mgrp import Constraints

constraints = (
    Constraints()
    .splits(strength=0.5, admin_col="COUNTY")
    .group_hinge(
        strength=1.0,
        group_pop_col="BVAP",
        total_pop_col="VAP",
        targets=[0.40, 0.50],
    )
)

constrained_run = SMCRunInfo(
    pop_col="TOTPOP",
    n_dists=14,
    n_sims=1_000,
    constraints=constraints,
    tally_columns=["TOTPOP", "VAP", "BVAP"],
    writer="ben",
    rng_seed=2026,
)

`group_hinge()` selects the nearest target independently for each district; the list does not
require a particular number of districts to reach each target. `splits()` counts administrative
units split at least once, not the number of pieces. See the [MGRP API](../../api/mgrp.rst) for
the exact formulas and parameters.

## Output files and tallies

| Writer | Primary output | Additional files |
| --- | --- | --- |
| `jsonl` | Standard assignment records | Metadata; tally CSV when columns are requested |
| `ben` | Compact BEN assignment stream | Metadata; tally CSV when columns are requested |
| `csv` | Native `redist` plans CSV | Metadata and a separate assignments CSV |

With `jsonl` or `ben`, `tally_columns` are written to `<output stem>_tallies.csv`; they are not
embedded in each assignment record. With `csv`, the tallies remain in the plans CSV and the
assignments receive their own sidecar. `expected_files()` reports the complete set for a run.

SMC does not support `mcmc_run_with_updaters()` because it is not an MCMC assignment stream.
Write the ensemble, then use the scoring workflow for additional plan metrics.

## Inspect the effective run

The effective configuration separates the `redist_map()` fields from the sampler fields and
includes the normalized constraints and mounted container paths. Inspecting it before the run
is often the clearest way to confirm which settings will reach R.

In [ ]:
config.run_config(constrained_run)

## Related

- [Ensemble runners overview](../mgrp.md)
- [Rust ReCom](recom.ipynb)
- [Forest ReCom](forest.ipynb)
- [Scoring](../scoring/index.md)
- [MGRP API](../../api/mgrp.rst)